# 使用@tool装饰器
自动将普通 Python 函数转化为智能体可调用的工具。
## 自定义工具描述:description
### 仅提供docstring信息
在bind_tools()调用时，先将函数封装为 BaseTool 类型的对象，再传递给convert_to_openai_tool 函数，生成工具的描述。
@tool 会从 docstring 生成描述信息，同样要求遵循 Google docstring 规范 。如果没有 docstring则报错。

In [1]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool  # 从 core 导入，避免 langgraph 版本冲突
from rich import print as rprint

@tool
def get_weather(city: str ='杭州'):
    """
    天气查询工具
    Args:
        city: 城市名称
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具\nArgs:\n    city: 城市名称',
        'parameters': {'properties': {'city': {'default': '杭州', 'type': 'string'}}, 'type': 'object'}
    }
}

### 添加工具描述description
@tool的参数description可以更改工具描述，优先级高于docstring的函数说明

In [9]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint

@tool(description="根据城市名称查询当日天气的工具")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '根据城市名称查询当日天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

### 解析docstring:parse_docstring
当我们没有向 @tool 传递 `description` 参数时，默认情况下， tool 会将 `docstring` 整体视为`description` 通过将 `parse_docstring` 设置为`True`，docstring会被解析，填充到相应的字段描述中.

In [28]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool
from rich import print as rprint

@tool(description="这是一个获取当日天气的工具",parse_docstring=True)
def get_weather(
    city: str,
    units: str = "celsius",
    include_forecast: bool = False,
) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报

    Args:
        city: 城市名称
        units: 气温单位，可选 celsius（摄氏度）或 fahrenheit（华氏度）
        include_forecast: 是否包含未来五日的天气预报
    """
    temp = 22 if units == "celsius" else 72
    result = f"{city}当天气温: {temp} {'摄氏度' if units == 'celsius' else '华氏度'}"
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '这是一个获取当日天气的工具',
        'parameters': {
            'properties': {
                'city': {'description': '城市名称', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，可选 celsius（摄氏度）或 fahrenheit（华氏度）',
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

### 更改工具名称：name_or_callable
默认情况，使用函数名作为工具名称，但可以向@tool 传参 name_or_callable ，以更改工具名称。@tool中参数name_or_callable名称可以省略

In [30]:
@tool(name_or_callable="getWeather")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 自定义args_schema
### 使用Pydantic模型定义
优势在于能够精确控制工具参数的格式和验证规则，让大模型更准确地理解如何调用工具。
#### BaseModel基类
继承核心基类BaseModel定义数据模型，从而声明字段结构、类型约束、默认值以0及校验规则。

In [32]:
from pydantic import BaseModel

# 子类WeatherInput继承BaseModel
class WeatherInput(BaseModel):
    city: str

@tool(args_schema=WeatherInput)
def get_weather(city):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

#### Field
`Field()`：用来`定制字段`的函数，可用于设置默认值、描述等

In [ ]:
# 设置默认值
from pydantic import BaseModel, Field
class WeatherInput(BaseModel):
    city: str = Field(
        default= "北京"
    )
print(WeatherInput())

@tool(args_schema=WeatherInput)
def get_weather(city):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

city='北京'


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'default': '北京', 'type': 'string'}}, 'type': 'object'}
    }
}

In [35]:
# 设置参数的描述信息
## 每个字段的 description 参数至关重要，它直接影响大模型理解参数含义的能力。
from pydantic import BaseModel, Field

class WeatherInput(BaseModel):
    city: str = Field(
        default= "北京",
        description="城市"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报"
    )
print(WeatherInput())

@tool(args_schema=WeatherInput)
def get_weather(city, include_forecast=False):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

city='北京' include_forecast=False


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '城市', 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日天气预报',
                    'type': 'boolean'
                }
            },
            'type': 'object'
        }
    }
}

#### Literal
可以使用 Literal类型限定参数为固定选项。

Literal ：表示字段不能是任意某种类型的值，而只能是几个固定字面量之一。

In [ ]:
from pydantic import BaseModel
from typing import Literal

class WeatherInput(BaseModel):
    city: str
    unit: Literal["celsius", "fahrenheit"]
print("===============> 合法 <===============")
print(WeatherInput(city="北京", unit="celsius"))
print("===============> 非法 <===============")
try:
    print(WeatherInput(city="北京", unit="kelvin"))
except Exception as e:
    print("报错类型：", type(e).__name__)
    print(e)

In [41]:
from pydantic import BaseModel, Field
from typing import Literal
class WeatherInput(BaseModel):
    city: str = Field(
        default= "北京",
        description="城市"
    )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="气温单位"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报"
    )
print(WeatherInput())

@tool(args_schema=WeatherInput)
def get_weather(city, unit,include_forecast):
    """
    天气查询工具
    """
    return f"{city}天气晴朗,{unit},{include_forecast}"


rprint(convert_to_openai_tool(get_weather))

city='北京' unit='celsius' include_forecast=False


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '天气查询工具',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '城市', 'type': 'string'},
                'unit': {
                    'default': 'celsius',
                    'description': '气温单位',
                    'enum': ['celsius', 'fahrenheit'],
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日天气预报',
                    'type': 'boolean'
                }
            },
            'type': 'object'
        }
    }
}

### 方式2：使用Json Schema定义
直接使用 JSON Schema 字典 来定义工具的参数模式。这种方式提供了极大的灵活性。
因为工具参数模式可以基于数据库配置或用户输入在 `运行时动态生成` ，所以这种方式特别适合参数结构需要动态生成的场景。

In [ ]:
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "获取当日天气，可选未来五日天气预报",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string"},
                "units": {"type": "string"},
                "include_forecast": {"type": "boolean"}
            },
            "required": ["location", "units", "include_forecast"]
        },
    }
}

In [43]:
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"}
    },
    "required": ["location", "units", "include_forecast"]
}

@tool(args_schema=weather_schema)
def get_weather(city: str, unit: str = "celsius",include_forecast: bool = False) -> str:
    """获取当日天气，可选未来五日天气预报"""
    temp = 22 if unit == "celsius" else 72
    result = f'{city}当天气温: {temp} {"摄氏度" if unit == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选未来五日天气预报',
        'parameters': {
            'type': 'object',
            'properties': {
                'location': {'type': 'string'},
                'units': {'type': 'string'},
                'include_forecast': {'type': 'boolean'}
            },
            'required': ['location', 'units', 'include_forecast']
        }
    }
}